# UrduHaluBench — run the open models (Qwen3-8B, Llama-3.1-8B)

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Copy your repository to Google Drive as `MyDrive/UrduHaluBench` (or clone it there; at least `scripts/` and `data/benchmark/benchmark_v1.csv` are needed).
3. For Llama: open https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, log in and click **Agree / request access** (usually approved within minutes). Then create a *read* token at https://huggingface.co/settings/tokens.

Answers are written straight to `MyDrive/UrduHaluBench/data/generations/`. If Colab disconnects, just run the cells again — the script continues where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/UrduHaluBench
!ls data/benchmark

In [ ]:
!pip -q install -U bitsandbytes accelerate transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Hugging Face login (paste your token when asked) - needed for Llama
from huggingface_hub import login
login()

## 1. Pilot first (30 items × 2 languages = 60 answers per model, ~5 min each)

In [ ]:
!python scripts/03_generate.py --backend hf --model Qwen/Qwen3-8B --name qwen3-8b --pilot

In [ ]:
import pandas as pd
d = pd.read_csv('data/generations/qwen3-8b.csv', keep_default_na=False)
print((d.error != '').sum(), 'errors')
d[['item_id', 'language', 'answer']].head(10)

Look at a few answers above. If they look sensible (short, in the right language, no `<think>` text), continue.

In [ ]:
!python scripts/03_generate.py --backend hf --model meta-llama/Llama-3.1-8B-Instruct --name llama3.1-8b --pilot

## 2. Full benchmark (350 items × 2 languages = 700 answers per model, roughly 1 hour each on a T4)
Run one model per Colab session if needed. Restart the runtime between models to free GPU memory (`Runtime → Restart session`, then re-run the first three cells).

In [ ]:
!python scripts/03_generate.py --backend hf --model Qwen/Qwen3-8B --name qwen3-8b

In [ ]:
!python scripts/03_generate.py --backend hf --model meta-llama/Llama-3.1-8B-Instruct --name llama3.1-8b

## 3. Done
The answers are in `data/generations/qwen3-8b.csv` and `llama3.1-8b.csv` on Drive; copy them into the same folder of your local clone.